In [ ]:
"""Setup: load YAML data + flat alt_df, derive helper bindings used by every chart cell.

The shared module `prompt_analysis.py` lives next to this notebook in the project root.
"""
import importlib
import altair as alt
import pandas as pd

import prompt_analysis
importlib.reload(prompt_analysis)   # pick up edits without restarting the kernel
from prompt_analysis import (
    load_yaml, build_alt_df, version_order, category_colors,
    directiveness,
    SR_CLASS_COLORS, SENT_REGISTER_CLASSES, TABLEAU10,
)

alt.data_transformers.disable_max_rows()

data              = load_yaml()                  # default: prompt_linguistic_analysis.yaml
alt_df            = build_alt_df(data)
by_category       = data["by_category"]
corpus_block      = data["corpus"]
per_file_records  = data["files"]
cats              = list(by_category.keys())
VOCAB_KEYS        = list(data["lexicons"]["VOCAB"].keys())

# Composite directiveness column — used by 05 and 06.
alt_df["directiveness"] = directiveness(alt_df)

# Per-category palette + Altair encodings used across charts.
CATEGORY_COLORS = category_colors(cats)
_cat_domain     = cats
_cat_range      = [CATEGORY_COLORS[c] for c in cats]

print(f"loaded {len(per_file_records)} files | {alt_df.shape[1]} columns | {len(cats)} categories | {len(VOCAB_KEYS)} VOCAB keys")


# Trends across `ccVersion` (Claude Code releases)

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code release the prompt was last touched in. Three temporal views:

1. **Per-file directiveness over `ccVersion`** — jittered scatter: every file as one point, sized by token count, y-axis is the extended directiveness composite z-score. Below it, a stacked-by-category histogram of how many files exist per minor version (2.0, 2.1).

2. **Loudness & imperative density across `ccVersion`** — four small multiples plotting per-`ccVersion` mean of: ALL CAPS density, CAPS-imperative density, imperative-marker density (per word), imperative-sentence share (% of sentences). Independent y-axes.

3. **Sentence-register classes across `ccVersion`** — six small multiples plotting per-`ccVersion` mean `sent_pct` of each pragmatic register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (tiny) trends without being squashed under the `imperative` scale.


### ccVersion timeline

Each prompt's `ccVersion` (from the HTML-comment frontmatter) records the Claude Code
release the prompt was last touched in. The chart below puts every file at its
`ccVersion` on the x-axis and a directiveness signal on the y-axis. Hover any point for
the prompt's full `name`, `description`, and version. Brush horizontally to focus on a
release window; click a category in the legend to highlight just that category.

In [ ]:
"""ccVersion timeline + minor-version histogram, with name/description tooltips.

Directiveness composite uses the extended formula from cell 58 (mood markers +
hard prohibitions + CAPS imperatives + directive_sent_pct + configuring_sent_pct
- collaborative_sent_pct - permissive_sent_pct - appreciative_sent_pct).
"""

# Sort ccVersion values numerically (treating "2.1.53" as a tuple)
version_order = (
    alt_df[alt_df["ccVersion"] != ""]
    .drop_duplicates("ccVersion")
    .sort_values("ccVersion_sort")
    ["ccVersion"]
    .tolist()
)

minor_counts = (
    alt_df[alt_df["ccMinor"] != "unknown"]
    .groupby(["ccMinor", "category"])
    .size()
    .reset_index(name="count")
)

def _minor_key(m):
    parts = m.split(".")
    try:
        return (int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)
    except (ValueError, IndexError):
        return (-1, -1)
minor_order = sorted(minor_counts["ccMinor"].unique(), key=_minor_key)

cat_color = alt.Color("category:N",
                       scale=alt.Scale(domain=_cat_domain, range=_cat_range),
                       legend=alt.Legend(title="Category", orient="bottom", columns=4))
legend_sel = alt.selection_point(fields=["category"], bind="legend")

# Use the precomputed `directiveness` from cell 58 — same formula.
df_with_ver = alt_df[alt_df["ccVersion"] != ""].copy()

timeline = (
    alt.Chart(df_with_ver)
    .mark_circle(opacity=0.65)
    .encode(
        x=alt.X("ccVersion:N", sort=version_order, title="ccVersion (oldest → newest)",
                axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
        y=alt.Y("directiveness:Q", title="Composite directiveness (z-score, extended)"),
        size=alt.Size("n_tokens:Q", scale=alt.Scale(range=[20, 400]), legend=None),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.8), alt.value(0.07)),
        xOffset="jitter:Q",
        tooltip=[
            alt.Tooltip("name:N",        title="Name"),
            alt.Tooltip("description:N", title="Description"),
            alt.Tooltip("ccVersion:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("n_tokens:Q",    format=","),
            alt.Tooltip("directiveness:Q", format=".2f"),
            alt.Tooltip("path:N"),
        ],
    )
    .transform_calculate(jitter="random()-0.5")
    .add_params(legend_sel)
    .properties(width=820, height=320,
                title="Per-file directiveness over ccVersion (jittered, hover for prompt name)")
)

hist = (
    alt.Chart(minor_counts)
    .mark_bar()
    .encode(
        x=alt.X("ccMinor:N", sort=minor_order,
                title="ccVersion minor (major.minor)"),
        y=alt.Y("count:Q", title="files"),
        color=cat_color,
        opacity=alt.condition(legend_sel, alt.value(0.95), alt.value(0.1)),
        tooltip=[
            alt.Tooltip("ccMinor:N"),
            alt.Tooltip("category:N"),
            alt.Tooltip("count:Q", title="files"),
        ],
    )
    .properties(width=820, height=180,
                title="Files per Claude Code minor version, stacked by category")
)

timeline & hist


### Loudness & imperative density across ccVersion

Four small multiples plotting the per-`ccVersion` mean of:

- **Loudness** — ALL CAPS density and CAPS imperative density (both % of file tokens).
- **Imperatives** — imperative-marker density (% of file tokens) and the share of sentences classified as imperative (% of sentences).

Each panel uses an independent y-axis so the absolute scales of the four metrics don't squash each other. Versions are ordered oldest → newest along x.

In [ ]:
"""Loudness & imperative-density trends across ccVersion."""

df_ver = alt_df[alt_df["ccVersion"] != ""].copy()

trend_specs = [
    ("all_caps_pct",        "Loudness — ALL CAPS density",                         "% of file tokens", "#e15759"),
    ("caps_imp_pct",        "Loudness — CAPS imperative density",                  "% of file tokens", "#af7aa1"),
    ("mood_marker_pct",     "Imperatives — imperative-marker density (per word)",  "% of file tokens", "#4e79a7"),
    ("imperative_sent_pct", "Imperatives — imperative sentences (per sentence)",   "% of sentences",   "#f28e2c"),
]

frames = []
for col, label, unit, _ in trend_specs:
    g = (
        df_ver.groupby(["ccVersion", "ccVersion_sort"])[col]
        .mean()
        .reset_index()
        .rename(columns={col: "value"})
    )
    g["metric"] = label
    g["unit"] = unit
    frames.append(g)
trend_df = pd.concat(frames, ignore_index=True)


def _trend_panel(label, unit, color, show_x_title):
    return (
        alt.Chart(trend_df[trend_df["metric"] == label])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
                   strokeWidth=2.5, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion (oldest → newest)" if show_x_title else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
            y=alt.Y("value:Q", title=unit),
            tooltip=[
                alt.Tooltip("ccVersion:N"),
                alt.Tooltip("value:Q", format=".3f", title="mean"),
                alt.Tooltip("unit:N"),
            ],
        )
        .properties(width=820, height=150, title=label)
    )


panels = [
    _trend_panel(label, unit, color, show_x_title=(i == len(trend_specs) - 1))
    for i, (_, label, unit, color) in enumerate(trend_specs)
]

alt.vconcat(*panels).resolve_scale(y="independent")


### Sentence-register classes across ccVersion

Six small multiples plotting the per-`ccVersion` mean of each sentence-register class. Independent y-axes so the near-zero `collaborative` and `appreciative` panels still show their (small) trends without being squashed under the imperative scale.

**Rationale for keeping near-zero classes visible**: knowing what's absent across ccVersion is part of the picture. If the corpus ever starts using interpersonal or appreciative speech, the line will lift off zero — that lift is the signal worth catching.

In [ ]:
"""Sentence-register classes across ccVersion — 6-panel small-multiples."""

sr_trend_specs = [
    ("collaborative", "Collaborative — interpersonal 1p-plural", "#4e79a7"),
    ("permissive",    "Permissive — soft-directive permission",  "#76b7b2"),
    ("appreciative",  "Appreciative — gratitude / praise",       "#59a14f"),
    ("imperative",    "Imperative — grammatical mood",           "#e15759"),
    ("directive",     "Directive — must / should / never markers", "#f28e2c"),
    ("configuring",   "Configuring — config / parameter speech", "#af7aa1"),
]

sr_frames = []
for cls, _label, _color in sr_trend_specs:
    col = f"{cls}_sent_pct"
    g = (
        df_with_ver.groupby(["ccVersion", "ccVersion_sort"])[col]
        .mean()
        .reset_index()
        .rename(columns={col: "value"})
    )
    g["class"] = cls
    sr_frames.append(g)
sr_trend_df = pd.concat(sr_frames, ignore_index=True)


def _sr_panel(cls, label, color, show_x_title):
    return (
        alt.Chart(sr_trend_df[sr_trend_df["class"] == cls])
        .mark_line(point=alt.OverlayMarkDef(filled=True, size=60),
                   strokeWidth=2.5, color=color)
        .encode(
            x=alt.X("ccVersion:N", sort=version_order,
                    title="ccVersion (oldest → newest)" if show_x_title else None,
                    axis=alt.Axis(labelAngle=-90, labelLimit=80, labelOverlap=False)),
            y=alt.Y("value:Q", title="% of sentences"),
            tooltip=[
                alt.Tooltip("ccVersion:N"),
                alt.Tooltip("value:Q", format=".3f", title="mean sent_pct"),
                alt.Tooltip("class:N"),
            ],
        )
        .properties(width=820, height=130, title=label)
    )


sr_panels = [
    _sr_panel(cls, label, color, show_x_title=(i == len(sr_trend_specs) - 1))
    for i, (cls, label, color) in enumerate(sr_trend_specs)
]

alt.vconcat(*sr_panels).resolve_scale(y="independent")
